# Hour-of-day distribution — lockdown vs. after (week 4, distributions)

Builds the Analysis 8 chart from `analysis-log.md` — the parked candidate from
Analysis 1's Stage 4 ("hour-of-day distribution, lockdown vs. outside — a
genuinely separate question, daily rhythm not overall trend"), now built for
the week-4 distributions assignment
(`notebooks/analysis/week4-distribution-requirements.md`).

**Proposition (Stage 1):** the hour-of-day distribution of messages shifts
between the lockdown period(s) and the post-lockdown period — during
lockdown, daytime and late-night hours were expected to gain share at the
expense of the typical evening peak; after lockdown, the shape was expected
to revert toward a more structured, evening-heavy pattern.

Loads the same already-anonymised, already-featured export as every other
analysis notebook — no re-parsing here.


In [ ]:
import numpy as np
import pandas as pd

from goad_toolkit.datatransforms import FlagDates, Pipeline, TimeFeatures
from goad_toolkit.visualizer import GroupedBarPlot, LinePlot, HorizontalLine, FacetPlot, PlotSettings

from wa_analyzer.data import load_own_chat

own = load_own_chat()
# same fix as 03.3-events-in-your-chat.ipynb: this processed file stores `timestamp`
# as plain strings, not datetime -- coerce once, here, for every cell downstream.
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)
own = Pipeline().add(TimeFeatures, column="timestamp", features=["hour"]).apply(own)
own.shape


## Reference periods

Same lockdown lookup as Analysis 1 (`analysis-log.md` Stage 2), but only the
two actual lockdown windows — Stage 2 of this analysis explicitly chose to
pool both as `during` rather than use only the first/longer one, and the
"partial tightening" window is not a lockdown so it stays out of this
comparison. `after` is everything strictly after the last lockdown ends;
rows before the first lockdown, and rows in the gap between the two
lockdowns, are excluded from this two-group comparison (they're neither
`during` nor `after` in the sense this chart is testing). Stage 3's
sensitivity check confirmed the exact `after` start date doesn't change the
read, so the simplest cutoff is used here.


In [ ]:
LOCKDOWN_WINDOWS = [
    ("Lockdown 2020-2021", "2020-12-14", "2021-04-28"),
    ("Lockdown 2021-2022", "2021-12-19", "2022-01-14"),
]
LAST_LOCKDOWN_END = pd.Timestamp("2022-01-14")

lockdown_dates = pd.concat(
    [pd.Series(pd.date_range(start, end)) for _, start, end in LOCKDOWN_WINDOWS]
)
own = Pipeline().add(
    FlagDates, column="timestamp", dates=lockdown_dates, feature="is_lockdown"
).apply(own)

own["period"] = np.select(
    [own["is_lockdown"], own["timestamp"] > LAST_LOCKDOWN_END],
    ["during", "after"],
    default=None,
)
own["period"].value_counts(dropna=False)


## Hour-of-day shares, every hour forced onto the axis

Requirement 1 of the assignment: every possible hour (0-23) must appear even
at zero/near-zero count — a dice-roll axis, not just the hours that happened
to occur. `value_counts` alone would silently drop an hour with zero
messages in one period, so the share is reindexed against the full 0-23
range per period, filling any missing hour with 0.


In [ ]:
tagged = own.dropna(subset=["period"])

full_index = pd.MultiIndex.from_product(
    [["during", "after"], range(24)], names=["period", "hour"]
)
counts = tagged.groupby(["period", "hour"]).size().rename("n").reset_index()
counts["share"] = counts["n"] / counts.groupby("period")["n"].transform("sum")
shares = (
    counts.set_index(["period", "hour"])["share"]
    .reindex(full_index, fill_value=0.0)
    .rename("share")
    .reset_index()
)
shares["share_pct"] = shares["share"] * 100
shares.head()


## Primary chart: likelihood of each hour, during vs. after

Requirement 2: y-axis is likelihood (share of messages), not raw count —
`during` (112 days, 2,049 messages) and `after` (1,088 days, 16,469
messages) are pooled to entirely different totals, so a raw-count
comparison would be meaningless on its own.

**The single comparison this plot exists to make:** does the hour-of-day
shape differ between the two periods, and specifically at which hours.


In [ ]:
DURING_COLOR = "#C44E52"
AFTER_COLOR = "#4C72B0"

settings = PlotSettings(
    figsize=(11, 5),
    title="Hour-of-day likelihood, during lockdown vs. after",
    xlabel="hour of day",
    ylabel="share of messages (%)",
    legend_title="period",
)
bars = GroupedBarPlot(settings)
fig, ax = bars.plot(
    data=shares, x="hour", y="share_pct", hue="period",
    order=list(range(24)), hue_order=["during", "after"],
    palette={"during": DURING_COLOR, "after": AFTER_COLOR},
)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("hour-of-day-lockdown-vs-after.png", dpi=150)


## Companion chart: the shift itself (during minus after)

Same idiom as `03.3-events-in-your-chat.ipynb`'s residual move
(`SubtractBaseline` + a zero line) — makes requirement 3 (a *shift* tied to
an event) the direct subject of a chart, rather than something the reader
has to compute by eye from two overlaid bars.

Note: this is a supporting/companion view of the shift, not a distribution to read on its own — the grouped-bar chart above is the one to look at for "what does the distribution look like."

In [ ]:
wide = shares.pivot(index="hour", columns="period", values="share_pct")
diff = (wide["during"] - wide["after"]).rename("diff_pct").reset_index()

diff_settings = PlotSettings(
    figsize=(9, 4),
    title="Hour-of-day share, during lockdown minus after (percentage points)",
    xlabel="hour of day",
    ylabel="during minus after (pp)",
)
lines = LinePlot(diff_settings)
fig, ax = lines.plot(data=diff, x="hour", y="diff_pct", marker="o", color="#333333")
lines.plot_on(HorizontalLine(diff_settings), y=0)
ax.set_xticks(range(0, 24, 2))
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("hour-of-day-lockdown-diff.png", dpi=150)


## Robustness check: is this 1-2 people, or the whole group?

Stage 2/3's pseudoreplication concern — message share across the 9 authors
ranges 5.1%-15.3%, "fairly evenly active" roughly holds, but that was on
total share, not on the actual per-author hour shape. Same per-author small
multiples pattern as Analysis 1's companion chart. Each author's shares are
normalized within their own during/after totals so a chattier person doesn't
dominate the panel's y-scale differently — `sharey=True` still puts every
panel on the same scale so the shapes compare honestly.

Same note: this is a robustness check on the primary claim, not itself meant to be read as a distribution — 9 small panels are for confirming the pattern generalizes, not for estimating a shape from.

In [ ]:
author_full_index = pd.MultiIndex.from_product(
    [tagged["author"].unique(), ["during", "after"], range(24)],
    names=["author", "period", "hour"],
)
author_counts = tagged.groupby(["author", "period", "hour"]).size().rename("n").reset_index()
author_counts["share"] = author_counts["n"] / author_counts.groupby(["author", "period"])["n"].transform("sum")
by_author = (
    author_counts.set_index(["author", "period", "hour"])["share"]
    .reindex(author_full_index, fill_value=0.0)
    .rename("share")
    .reset_index()
)
by_author["share_pct"] = by_author["share"] * 100
# `FacetPlot.plot`'s own `order=` controls panel order (author), so hour order
# has to be fixed via dtype instead of an `order=` kwarg meant for GroupedBarPlot.
by_author["hour"] = pd.Categorical(by_author["hour"], categories=list(range(24)), ordered=True)

facet_settings = PlotSettings(
    figsize=(14, 10),
    title="Hour-of-day likelihood per author, during vs. after",
    ylabel="share (%)",
    sharey=True,
    max_cols=3,
    legend_title="period",
)
facet = FacetPlot(facet_settings)
fig, axes = facet.plot(
    inner=GroupedBarPlot(facet_settings), data=by_author, by="author",
    x="hour", y="share_pct", hue="period", hue_order=["during", "after"],
    palette={"during": DURING_COLOR, "after": AFTER_COLOR},
)
fig.tight_layout()
fig.savefig("hour-of-day-lockdown-per-author.png", dpi=150)


## Spike-day check: is hour 21's jump one day, or the whole `during` window?

Stage 5 honesty check, run before trusting the hour-21 spike as a real
rhythm shift rather than 1-2 loud days (same "label spike days, don't let
them masquerade as the finding" precedent as Analysis 1).


In [ ]:
during = tagged[tagged["period"] == "during"].copy()
during["date"] = during["timestamp"].dt.date
hour21 = during[during["hour"] == 21]
by_date = hour21.groupby("date").size().sort_values(ascending=False)
print(f"hour==21 during lockdown: {len(hour21)} messages across {by_date.shape[0]} distinct days")
print(f"busiest single day contributes {by_date.iloc[0]} messages "
      f"({by_date.iloc[0] / len(hour21):.1%} of hour-21-during total)")
by_date.head(5)


## Correction: excluding election days from *both* sides

Named after reacting to the spike-day check: the single during-lockdown
election (2021-03-17, NL general) explains 35.5% of the hour-21 during-count
— but this project's own election table (Analysis 3/4/6) lists 5 elections
total, and 3 of them (2023-11-22, 2024-11-05, 2025-10-29) fall inside the
`after` window too. If election nights generically spike hour 21, `after`
already absorbs that effect (diluted over far more days), so excluding
election days from only `during` would be an unfair, one-sided correction.
Checked which of the 5 election dates actually have messages in this chat:
2021-03-17 (173 msgs, in `during`) and 2023-11-22 (68 msgs, in `after`) do;
2024-11-05 and 2025-10-29 have zero messages that exact day. Both real
election days are excluded below, one from each period.


In [ ]:
ELECTION_DAYS = pd.to_datetime(
    ["2020-11-03", "2021-03-17", "2023-11-22", "2024-11-05", "2025-10-29"]
).date

tagged_excl = tagged[~tagged["timestamp"].dt.date.isin(ELECTION_DAYS)]

counts_excl = tagged_excl.groupby(["period", "hour"]).size().rename("n").reset_index()
counts_excl["share"] = counts_excl["n"] / counts_excl.groupby("period")["n"].transform("sum")
shares_excl = (
    counts_excl.set_index(["period", "hour"])["share"]
    .reindex(full_index, fill_value=0.0)
    .rename("share")
    .reset_index()
)
shares_excl["share_pct"] = shares_excl["share"] * 100

wide_excl = shares_excl.pivot(index="hour", columns="period", values="share_pct")
diff_excl = (wide_excl["during"] - wide_excl["after"]).round(2)
print("hour  during  after  diff (election days excluded from both sides)")
print(pd.DataFrame({
    "during": wide_excl["during"].round(2),
    "after": wide_excl["after"].round(2),
    "diff": diff_excl,
}).loc[[19, 20, 21, 13, 10, 0]])


**Read:** hour 21's gap collapses from +4.3pp to about +1pp once the
election days are excluded from both sides — most of the original hour-21
"during lockdown" spike was the 2021 election, not a lockdown effect.
**Hour 19 survives essentially unchanged** (+2.9pp before, ~+3.0pp after) —
that one is not an election artifact. **Revised headline:** the lockdown
effect on hour-of-day is real but narrower than first read — centered on
hour 19, not on both 19 and 21.


In [ ]:
settings_v2 = PlotSettings(
    figsize=(11, 5),
    title="Hour-of-day likelihood, during lockdown vs. after (election days excluded)",
    xlabel="hour of day",
    ylabel="share of messages (%)",
    legend_title="period",
)
bars_v2 = GroupedBarPlot(settings_v2)
fig, ax = bars_v2.plot(
    data=shares_excl, x="hour", y="share_pct", hue="period",
    order=list(range(24)), hue_order=["during", "after"],
    palette={"during": DURING_COLOR, "after": AFTER_COLOR},
)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("hour-of-day-lockdown-vs-after-v2.png", dpi=150)


## Stage 6 — Verification: is the hour-19 gap more than chance?

A ~3pp gap in a chat this noisy could be luck. Same check as the
friends'-weekend analysis: `NullDistribution` shuffles the `during`/`after`
label across days, recomputes the gap a couple thousand times, and shows
where the real value sits in that cloud — the cheapest honest check that
assumes nothing about the shape of the data. Unit of observation is the
**day** (not the message), matching Stage 2/3's independent-units decision —
112 `during` days vs. 1,087 `after` days (election day already excluded from
each), each day's own share of its messages sent at hour 19.


In [ ]:
from goad_toolkit.analytics import NullDistribution
from goad_toolkit.visualizer import NullPlot

daily_totals = tagged_excl.groupby(["period", tagged_excl["timestamp"].dt.date]).size().rename("total")
daily_hour19 = (
    tagged_excl[tagged_excl["hour"] == 19]
    .groupby(["period", tagged_excl.loc[tagged_excl["hour"] == 19, "timestamp"].dt.date])
    .size()
    .rename("hour19")
)
days = pd.concat([daily_totals, daily_hour19], axis=1).fillna(0.0)
days["hour19_share"] = days["hour19"] / days["total"]
days = days.reset_index().rename(columns={"level_1": "date"})
days["is_during"] = days["period"] == "during"

print(f"{days['is_during'].sum()} during-days, {(~days['is_during']).sum()} after-days")


def hour19_gap(frame: pd.DataFrame) -> float:
    means = frame.groupby("is_during")["hour19_share"].mean()
    return means[True] - means[False]


null = NullDistribution(hour19_gap, n_iter=2000, seed=42)
result = null.run(days, label="is_during")

null_settings = PlotSettings(
    figsize=(8, 4),
    title="Hour-19 share, during minus after: is +gap more than chance?",
    xlabel="during − after, share of day's messages at hour 19",
)
fig, ax = NullPlot(null_settings).plot(result=result)
ax.legend()
fig.tight_layout()
fig.savefig("hour-of-day-lockdown-hour19-null.png", dpi=150)

print(f"observed gap: {result.observed:.4f}")
print(f"two-sided p: {result.p_value():.4f}   one-sided p (during higher): {result.p_value('greater'):.4f}")


**Read this against the null cloud, not just the p-value.** If the observed
gap sits comfortably inside the shuffled cloud, hour 19's shift is not
distinguishable from the chat's ordinary day-to-day noise, whatever the
exact p-value says — 112 `during`-days is a real but modest sample next to
1,087 `after`-days.


## Alternative sketched and set aside: month x hour heatmap

A month-by-month heatmap (share of each month's messages per hour, spanning
the Oct 2020 - May 2022 transition) was sketched as the "genuinely
different family" alternative this stage asks for — framing the shift as a
continuous transition in time rather than one before/after split. Rejected
as the primary or companion chart: with ~30 days per month split across 24
hours, single spike days (the same birthday/bad-news response spikes
Analysis 1 already flagged) dominate individual cells — e.g. one cell hit
33% of a month's messages, almost certainly one loud day, not a real
monthly pattern. Kept as a documented, rejected alternative rather than
rebuilt here, per the Analysis 1 Stage 4 precedent for alternatives that
don't survive contact with the data.

## Next: Stage 5 (critique)

Look at the three charts above the way `goad_analysis_checklist`'s Stage 5
table asks — before the next chat message answers its questions.
